# Multi-Agent Customer Support System using LangGraph

## Objective

Build a multi-agent customer support system capable of:

1. Accepting customer questions
2. Routing questions to specialized agents
3. Retrieving information from a knowledge base
4. Maintaining conversation memory
5. Performing quality control before responding

Technologies:

- LangGraph
- FAISS
- OpenAI
- Sentence Transformers

In [2]:
# !pip install langgraph
# !pip install langchain
# !pip install faiss-cpu
# !pip install sentence-transformers
# !pip install openai
# !pip install python-dotenv

In [3]:
import os
import faiss
import numpy as np

from dotenv import load_dotenv

from openai import OpenAI

from sentence_transformers import SentenceTransformer

from typing import TypedDict

from langgraph.graph import StateGraph
from langgraph.graph import END

In [4]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [5]:
technical_docs = [
    "Mudbox access requires manager approval.",
    "VPN access is mandatory for analytics systems."
]

product_docs = [
    "The budget product line is called BasicPlus.",
    "PremiumMax is the enterprise offering."
]

hr_docs = [
    "Employees may take sabbatical after five years.",
    "Parental leave is 16 weeks."
]


# # Install PDF library if not already installed
# # !pip install pypdf

# from pypdf import PdfReader

# # ------------------------------------------------------------------
# # Helper function to extract text from PDF
# # ------------------------------------------------------------------

# def extract_pdf_text(pdf_path):

#     reader = PdfReader(pdf_path)

#     text = ""

#     for page in reader.pages:

#         page_text = page.extract_text()

#         if page_text:
#             text += page_text + "\n"

#     return text


# # ------------------------------------------------------------------
# # Helper function to split text into chunks
# # ------------------------------------------------------------------

# def chunk_text(
#     text,
#     chunk_size=500
# ):

#     chunks = []

#     for i in range(
#         0,
#         len(text),
#         chunk_size
#     ):
#         chunks.append(
#             text[i:i + chunk_size]
#         )

#     return chunks


# # ------------------------------------------------------------------
# # File locations
# # ------------------------------------------------------------------

# BASE_PATH = r"C:\Users\deves\OneDrive\Documents\Gen AI Assignments\Week 4 - Agentic AI"

# technical_pdf = f"{BASE_PATH}\\technical.pdf"
# product_pdf = f"{BASE_PATH}\\product.pdf"
# hr_pdf = f"{BASE_PATH}\\hr.pdf"


# # ------------------------------------------------------------------
# # Extract text
# # ------------------------------------------------------------------

# technical_text = extract_pdf_text(
#     technical_pdf
# )

# product_text = extract_pdf_text(
#     product_pdf
# )

# hr_text = extract_pdf_text(
#     hr_pdf
# )


# # ------------------------------------------------------------------
# # Convert PDFs into chunks
# # ------------------------------------------------------------------

# technical_docs = chunk_text(
#     technical_text
# )

# product_docs = chunk_text(
#     product_text
# )

# hr_docs = chunk_text(
#     hr_text
# )


# # ------------------------------------------------------------------
# # Quick validation
# # ------------------------------------------------------------------

# print("Technical Chunks:", len(technical_docs))
# print("Product Chunks:", len(product_docs))
# print("HR Chunks:", len(hr_docs))

# print("\nSample Technical Chunk:\n")
# print(technical_docs[0][:500])

In [6]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
documents = (
    technical_docs
    + product_docs
    + hr_docs
)

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

In [8]:
class AgentState(TypedDict):
    question: str
    category: str
    context: str
    answer: str
    history: list

In [9]:
def router_agent(state):

    question = state["question"].lower()

    if any(
        word in question
        for word in [
            "server",
            "access",
            "vpn",
            "technical"
        ]
    ):
        category = "technical"

    elif any(
        word in question
        for word in [
            "product",
            "pricing",
            "budget"
        ]
    ):
        category = "product"

    else:
        category = "hr"

    return {
        **state,
        "category": category
    }

In [10]:
def retrieve_context(question):

    query_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        query_embedding,
        2
    )

    context = ""

    for idx in indices[0]:
        context += documents[idx]
        context += "\n"

    return context

In [11]:
def technical_agent(state):

    context = retrieve_context(
        state["question"]
    )

    return {
        **state,
        "context": context
    }

In [12]:
def product_agent(state):

    context = retrieve_context(
        state["question"]
    )

    return {
        **state,
        "context": context
    }

In [13]:
def hr_agent(state):

    context = retrieve_context(
        state["question"]
    )

    return {
        **state,
        "context": context
    }

In [14]:
def qa_agent(state):

    prompt = f"""
    Answer using context only.

    Context:
    {state['context']}

    Question:
    {state['question']}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    state["history"].append(
        {
            "question": state["question"],
            "answer": answer
        }
    )

    return {
        **state,
        "answer": answer
    }

In [15]:
graph = StateGraph(
    AgentState
)

graph.add_node(
    "router",
    router_agent
)

graph.add_node(
    "technical",
    technical_agent
)

graph.add_node(
    "product",
    product_agent
)

graph.add_node(
    "hr",
    hr_agent
)

graph.add_node(
    "qa",
    qa_agent
)

In [16]:
def route(state):

    return state["category"]

In [17]:
graph.set_entry_point(
    "router"
)

graph.add_conditional_edges(
    "router",
    route,
    {
        "technical":"technical",
        "product":"product",
        "hr":"hr"
    }
)

graph.add_edge(
    "technical",
    "qa"
)

graph.add_edge(
    "product",
    "qa"
)

graph.add_edge(
    "hr",
    "qa"
)

graph.add_edge(
    "qa",
    END
)

app = graph.compile()

In [18]:
result = app.invoke(
    {
        "question":
        "How can I get access to Mudbox?",
        "category":"",
        "context":"",
        "answer":"",
        "history":[]
    }
)

print(result["answer"])

To get access to Mudbox, you need to obtain manager approval.


In [19]:
result = app.invoke(
    {
        "question":
        "What is our budget product line?",
        "category":"",
        "context":"",
        "answer":"",
        "history":[]
    }
)

print(result["answer"])

The budget product line is called BasicPlus.


In [20]:
result = app.invoke(
    {
        "question":
        "What is the sabbatical policy?",
        "category":"",
        "context":"",
        "answer":"",
        "history":[]
    }
)

print(result["answer"])

Employees may take a sabbatical after five years.
